In [35]:
%load_ext autoreload
%autoreload 2

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display
import imageio.v2 as imageio
import utils as ut

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Interpreting k-space

In this section, I have written a quick explanation of how to interpret graphs in Fourier or k-space, which is relevant to the plot of structure factor of a lattice.

For simplicity, we consider only one particular row and we observe only the values of $\cos\theta$ across that row. However, the ideas are true regardless of the row or column we observe or of whether we look at sines or cosines.

At the start of the evolution, the lattice is generated with spins pointing in random directions, and hence the system is highly disordered.

<p align="center">
  <img src="images/structure_factor/random.png" alt="" width="400" align="center"/>
</p>

The crosses mark the values of $\cos\theta$ for each spin. If we connected the crosses to form a curve, this curve would have close to no periodicity. The Fourier transform (loosely speaking) expresses this curve as a (usually infinite) sum of sines and cosines of various frequencies and amplitudes. As there is no periodicity, the graph in k-space (a graph of amplitude vs $k_x$, the angular frequency of the sines/cosines) be spread across a large range of frequencies with no dominant spikes in amplitude (essentially noise).

After some time, perhaps the spins may settle into some sort of periodicity.

<p align="center">
  <img src="images/structure_factor/periodic.png" alt="" width="400" align="center"/>
</p>

The Fourier transform of this graph would have a sharp spike at

$$k_x = \pm\frac{2\pi}{\lambda \times a} = \pm\frac{2\pi}{4\times 1} = \pm\frac\pi 2,$$

where $\lambda$ is the wavelength of this curve and $a$ is the physical distance between lattice sites. To keep the simulation dimensionless, we take $a=1$, or alternatively, we measure wavelength as $\lambda\times a$ (ie in units of $a$). The graph in k-space would depict a large spike at $k_x = \pm\pi/2$ and perhaps some noise of tiny amplitude across all other frequencies.

If a ring is observed in the 2D plot, this would indicate that the lattice has some preference for a particular wavelength but no preferred direction.

After sufficient evolution, the system settles into an ordered state, where the spins are all pointing loosely in the same direction.

<p align="center">
  <img src="images/structure_factor/ordered.png" alt="" width="400" align="center"/>
</p>

This graph can be interpreted as a wave with zero frequency, and as such, the k-space graph would depict a large spike at $k_x=0$ and noise elsewhere. In the animation below, this is observed as the centre of the 2D k-space gradually increases in intensity as the evolution progresses.

# Structure factor

One spin $\vec s$ in the XY model is defined as

$$\vec s_i = (\cos\theta_i, \sin\theta_i).$$

The magnetic structure factor is given by

$$S(\vec k) = \frac1N \left|\sum_i\vec s_i e^{i\vec k\cdot\vec r_i}\right|^2.$$

Since my existing implementation already tracks the evolution of $\cos\theta_i$ and $\sin\theta_i$ of every spin $i$, the structure factor can be computed via

$$S(\vec k) = \frac 1 N \left(\left|\sum_i \cos\theta_i e^{i\vec k \cdot \vec r_i} \right|^2+\left|\sum_i\sin\theta_i e^{i\vec k\cdot\vec r_i}\right|^2\right)$$

In [ ]:
def structure_factor(lattice):
    N = lattice.size

    spin_x = np.cos(lattice)
    spin_y = np.sin(lattice)

    spin_x_k = np.fft.fft2(spin_x)
    spin_y_k = np.fft.fft2(spin_y)

    S_k = (np.abs(spin_x_k)**2 + np.abs(spin_y_k)**2) / N

    return np.fft.fftshift(S_k)

def make_structure_factor_gif(frames, filename="structure_factor_evolution.gif", fps=16):
    images = []
    L = frames.shape[1]

    kx = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(L))
    ky = 2 * np.pi * np.fft.fftshift(np.fft.fftfreq(L))

    # calculate all structure factors first
    structure_factors = np.array([structure_factor(frame) for frame in frames])

    # fixed colour scale for all frames
    plotted_S = np.log1p(structure_factors)

    vmin = plotted_S.min()
    vmax = plotted_S.max()

    for S_k in plotted_S:
        fig, ax = plt.subplots(figsize=(6, 6))
        im = ax.imshow(S_k, origin="lower", extent=[kx[0], kx[-1], ky[0], ky[-1]], aspect="equal", vmin=vmin, vmax=vmax)

        ax.set_xlabel(r"$k_x a$")
        ax.set_ylabel(r"$k_y a$")
        ax.set_title(r"$S(k_x,k_y)$")

        fig.colorbar(im, ax=ax, label=r"$\log(1 + S(\mathbf{k}))$")
        fig.tight_layout()
        fig.canvas.draw()
        image = np.frombuffer(fig.canvas.buffer_rgba(), dtype=np.uint8)
        image = image.reshape(fig.canvas.get_width_height()[::-1] + (4,))
        images.append(image.copy())
        plt.close(fig)

    imageio.mimsave(filename, images, fps=fps)

In [ ]:
L = 15 # number of spins in one side of the square lattice (so total there are N = L^2 spins)

# hamiltonian parameters
H = 0 # H = mu_s * B / J
phi = 0
BJ = 1.7

# metropolis parameters
delta = np.pi / 4 # the maximum a spin can rotate in one iteration
num_sweeps = 1000 # number of monte carlo sweeps = num_sweeps * N monte carlo iterations
save_every = L**2 # save snapshots of the lattice for animation every <save_every> monte carlo iteration

# initial portion of run to discard so only equilibrium states are considered for stats calculations
burn_in_fraction = 0.5 

# lattice spacing
a = 1

In [ ]:
# generate lattice of random spins
lattice = ut.generate_lattice(L)
energy = ut.get_energy(lattice, H, phi)

# execute algorithm
N = L**2
steps = N * num_sweeps
final_lattice, total_spins_x, total_spins_y, energies, frames = ut.metropolis(lattice, steps, BJ, H, phi, energy, delta, save_every)

In [42]:
mode = "arrows" if L <= 24 else "colours"
ut.make_gif(frames, f'images/structure_factor/real.gif', fps=32, mode=mode)

make_structure_factor_gif(frames, filename="images/structure_factor/k-space.gif", fps=32)

<p align="center">
  <img src="images/structure_factor/real.gif" width="45%">
  <img src="images/structure_factor/k-space.gif" width="45%">
</p>